In [146]:

# ANALISIS DE MÉTRICAS DE DIVERSIDAD
# Este script aplica un conjunto de enfoques complementarios para caracterizar
# la diversidad alfa en tu repertorio: primero calcula los números de Hill 
# mediante alphaDiversity de alakazam ; luego incorpora métricas adicionales 
# con vegan como índices de diversidad y equidad para describir la distribución
# de abundancias; y finalmente utiliza ineq para estimar desigualdad clonal a través
# del índice de Gini y otras medidas de concentración. Al combinar estas diez métricas, 
# obtienes una visión integrada de la cantidad, equilibrio y desigualdad en la arquitectura 
# clonal de tu repertorio.

In [147]:
# Paquetes y librerías
library(readr)
library(dplyr)
library(ggplot2)
library(viridisLite)
library(here)  # para rutas relativas
# install.packages("alakazam")
library(alakazam)
library(ineq)
library(vegan)
library(tidyr)
library(tibble)

In [148]:

# Cargar tu archivo .tsv
archivo_clones <- "../data/output/repertorio_D_insilico_10000_seqs_clone-pass.tsv"
clones <- read_tsv(archivo_clones)
# Agregar identificador de muestra (porque es simulado)
clones <- clones %>%
  mutate(sample_id = "repertorio_simulado")


# Contar secuencias por clon y muestra
clone_counts <- clones %>%
  group_by(sample_id, clone_id) %>%
  summarise(count = n(), .groups = "drop")

Rows: 10000 Columns: 50
-- Column specification --------------------------------------------------------
Delimiter: "\t"
chr (20): sequence_id, sequence, v_call, d_call, j_call, sequence_alignment,...
dbl (25): junction_length, np1_length, np2_length, v_sequence_start, v_seque...
lgl  (5): rev_comp, productive, stop_codon, vj_in_frame, c_call

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [149]:
hill <- alphaDiversity(
  data = clone_counts,
  clone = "clone_id",
  min_q = 0,
  max_q = 4,
  step_q = 1,
  nboot = 100,
  ci = 0.95
)

# Extraer la tabla
df <- hill@diversity

# Agregar columna con índices clásicos
df <- df %>%
  dplyr::mutate(
    indice_clasico = case_when(
      q == 0 ~ d,            # riqueza observada
      q == 1 ~ log(d),       # Shannon clásico H = ln(D1)
      q == 2 ~ 1/d,          # Simpson clásico D = 1/D2
      q == 3 ~ 1/(d^2),      # ∑ p_i^3 = 1/(D3^2)
      q == 4 ~ 1/(d^3)       # ∑ p_i^4 = 1/(D4^3)
    )
  )

print(df)
df_tbl <- as_tibble(df)

ERROR: Error: vector memory limit of 32.0 Gb reached, see mem.maxVSize()


In [ ]:
hill_numbers <- function(clones_df){
    clone_counts <- clones_df %>%
        group_by(sample_id, clone_id) %>%
        summarise(count = n(), .groups = "drop")
    hill <- alphaDiversity(
        data = clone_counts,
        clone = "clone_id",
        min_q = 0,
        max_q = 4,
        step_q = 1,
        nboot = 100,
        ci = 0.95
    )
    return(hill@diversity)
}
rep_hill_numbers <- hill_numbers(clones)
rep_hill_numbers

group,q,d,d_sd,d_lower,d_upper,e,e_lower,e_upper
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
All,0,968.1600,0.9069875,966.3823,969.9377,1.0000000,0.9981639,1.001836
All,1,967.8371,1.2547479,965.3778,970.2963,0.9996664,0.9971263,1.002207
All,2,967.3262,1.8033809,963.7917,970.8608,0.9991388,0.9954880,1.002790
All,3,966.5009,2.6856251,961.2372,971.7647,0.9982864,0.9928495,1.003723
All,4,965.1466,4.1211253,957.0693,973.2238,0.9968875,0.9885446,1.005230


In [ ]:
richness <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 0) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

richness(rep_hill_numbers)

[1] 968.16

In [ ]:
 q1 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q1(rep_hill_numbers)

[1] 967.8371

In [ ]:
shannon <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(log(metric_value))
} 

shannon(rep_hill_numbers)

[1] 6.875064

In [ ]:
 q2 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q2(rep_hill_numbers)

[1] 967.3262

In [ ]:
simpson <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value))
} 

simpson(rep_hill_numbers)

[1] 0.001033777

In [ ]:
 q3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 3) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q3(rep_hill_numbers)

[1] 966.5009

In [ ]:
d3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value)^2)
} 

d3(rep_hill_numbers)

[1] 1.067568e-06

In [ ]:
 q4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q4(rep_hill_numbers)

[1] 965.1466

In [ ]:
d4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value)^3)
} 

d4(rep_hill_numbers)

[1] 1.112296e-09

In [ ]:
# MÉTRICA CHAO1 y ACE PAQUETE VEGAN
metricas_chao1ace <- clone_counts %>%
  group_by(sample_id) %>% 
  summarise(
    chao1 = estimateR(count)["S.chao1"],
    ace   = estimateR(count)["S.ACE"]
  )

print(metricas_chao1ace)

# A tibble: 1 x 3
  sample_id            chao1    ace
  <chr>                <dbl>  <dbl>
1 repertorio_simulado 17349. 17847.


In [ ]:


chao1 <- function(clones_df){
  clone_counts <- clones_df %>%
    group_by(sample_id, clone_id) %>%
    summarise(count = n(), .groups = "drop")
  
  metricas_chao1 <- clone_counts %>%
    group_by(sample_id) %>%
    summarise(
      chao1 = as.numeric(vegan::estimateR(count)["S.chao1"]),
      .groups = "drop"
    )%>%
    dplyr::pull(chao1)
  
  return(metricas_chao1[1])
}

# Ejecutar
rep_chao1 <- chao1(clones)
print(rep_chao1)


[1] 17349.37


In [ ]:
ace <- function(clones_df){
  clone_counts <- clones_df %>%
    group_by(sample_id, clone_id) %>%
    summarise(count = n(), .groups = "drop")
  
  metricas_ace <- clone_counts %>%
    group_by(sample_id) %>%
    summarise(
      ace = as.numeric(vegan::estimateR(count)["S.ACE"]),
      .groups = "drop"
    )%>%
    dplyr::pull(ace)
  
  return(metricas_ace[1])
}

# Ejecutar
rep_ace <- ace(clones)
print(rep_ace)

[1] 17847.18


In [ ]:
# MÉTRICA GINI PAQUETE INEQ
calc_gini <- function(df) {
  ineq::ineq(df$count, type = "Gini")
}
gini_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(
    gini = calc_gini(cur_data())
  )

print(gini_result)

# A tibble: 1 x 2
  sample_id             gini
  <chr>                <dbl>
1 repertorio_simulado 0.0292


In [ ]:
gini <- function (clones_df){
    clone_counts <- clones_df %>%
        group_by(sample_id, clone_id) %>%
        summarise(count = n(), .groups = "drop")
    metrica_gini <- ineq::ineq(clone_counts$count, type = "Gini")
    return(metrica_gini)
}
gini(clones)

[1] 0.02921601

In [ ]:
# MÉTRICA PIELOU PAQUETE VEGAN

calc_pielou <- function(df) {
  abund <- df$count
  H <- diversity(abund, index = "shannon")  # Shannon
  S <- specnumber(abund)                    # número de clones
  J <- H / log(S)                           # Pielou
  return(J)
}

pielou_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(
    pielou = calc_pielou(cur_data())
  )

print(pielou_result)


# A tibble: 1 x 2
  sample_id           pielou
  <chr>                <dbl>
1 repertorio_simulado  0.998


In [ ]:
pielou <- function(clones_df){
 
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  H <- vegan::diversity(clone_counts$count, index = "shannon")
  S <- vegan::specnumber(clone_counts$count)
  J <- H / log(S)
  
  return(as.numeric(J))  # 👈 devuelve solo el número
}

pielou(clones)

[1] 0.9982275

In [ ]:
# MÉTRICA BASHARIN FUNCIONES R+ VEGAN

calc_basharin <- function(df) {
  abund <- df$count
  N <- sum(abund)
  S <- specnumber(abund)
  
  # Casos triviales
  if (N == 0 || S <= 1) return(0)
  
  H <- diversity(abund, index = "shannon")
  print(H)
  basharin <- H + (S - 1) / (2 * N)
  return(basharin)
}

# Aplicar por muestra
basharin_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(basharin = calc_basharin(cur_data()), .groups = "drop")

print(basharin_result)


[1] 6.864077
# A tibble: 1 x 2
  sample_id           basharin
  <chr>                  <dbl>
1 repertorio_simulado     7.35


In [ ]:
basharin <- function(clones_df){
  
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  abund <- clone_counts$count
  N <- sum(abund)
  S <- vegan::specnumber(abund)
  
  if (N == 0 || S <= 1) return(0)
  
  # Shannon
  H <- vegan::diversity(abund, index = "shannon")
  
  # Basharin
  basharin_val <- H + (S - 1) / (2 * N)
  
  return(as.numeric(basharin_val))  # 👈 devuelve número puro
}
basharin(clones)

[1] 7.348561

In [ ]:
d50_fun <- function(counts) {
  counts <- sort(counts, decreasing = TRUE)
  total <- sum(counts)
  cum <- cumsum(counts)
  which(cum >= 0.5 * total)[1]
}

# Calcular D50
d50_val <- d50_fun(clone_counts$count)
print(d50_val)
d50_result <- tibble::tibble(D50 = d50_val)


[1] 470


In [ ]:
d50 <- function(clones_df){
 
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  counts <- sort(clone_counts$count, decreasing = TRUE)
  total  <- sum(counts)
  cum    <- cumsum(counts)
  
  d50_val <- which(cum >= 0.5 * total)[1]
  
  return(as.numeric(d50_val))  # 👈 devuelve número puro
}
d50(clones)

[1] 470

In [ ]:
metricas_diversidad <- c(richness= richness(rep_hill_numbers),q1= q1(rep_hill_numbers),shannon= shannon(rep_hill_numbers), q2= q2(rep_hill_numbers), simpson= simpson(rep_hill_numbers), q3= q3(rep_hill_numbers), 
d3= d3(rep_hill_numbers), q4= q4(rep_hill_numbers), d4= d4(rep_hill_numbers), chao1= chao1(clones), gini= gini(clones), pielou= pielou(clones), basharin= basharin(clones), d50= d50(clones))
metricas_diversidad

richness           q1      shannon           q2      simpson           q3 
9.681600e+02 9.678371e+02 6.875064e+00 9.673262e+02 1.033777e-03 9.665009e+02 
          d3           q4           d4        chao1         gini       pielou 
1.067568e-06 9.651466e+02 1.112296e-09 1.734937e+04 2.921601e-02 9.982275e-01 
    basharin          d50 
7.348561e+00 4.700000e+02

In [ ]:
tabla_diversidad <- bind_rows(metricas_diversidad)
tabla_diversidad$sample_id <- "D_10000seq"
tabla_diversidad

richness,q1,shannon,q2,simpson,q3,d3,q4,d4,chao1,gini,pielou,basharin,d50,sample_id
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
968.16,967.8371,6.875064,967.3262,0.001033777,966.5009,1.067568e-06,965.1466,1.112296e-09,17349.37,0.02921601,0.9982275,7.348561,470,D_1000seq


In [ ]:
readr::write_tsv(tabla_diversidad, "../results/diversity_metrics/diversity_D_10000seqs.tsv")